# Visualisation de données
Ce notebook permet la visualisation de données afin de programmer correctement le fichier preprocessing.py. 

In [ ]:
# Import 
import os
import sys
import logging
import pandas as pd
import xgboost as xgb
from ydata_profiling import ProfileReport
import sweetviz as sv
from sklearn.model_selection import train_test_split


# 1. On nettoie les anciens verrous de logging spécifiques aux notebooks
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# 2. On configure proprement pour que ça print TOUT dans le notebook
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],  # <-- La magie est là, ça force l'affichage
)

# Reconstruire le chemin absolu à partir de la racine du container
path_data = os.path.join("/workspace", "data")
path_data_input = os.path.join("/workspace", "data", "input", "data_scoring_credit.csv")


## Analyse rapide

In [ ]:
# Test de lecture rapide du dataset
df = pd.read_csv(path_data_input)
logging.info(f"Dimensions du dataset : {df.shape}")
df.head()

### Dictionnaire des variables (Data Dictionary)

*   **`ncust`** : Numéro d'identifiant interne et court du client (à supprimer avant l'entraînement).
*   **`customer`** : Identifiant unique global du client dans le système bancaire (à supprimer avant l'entraînement).
*   **`branch`** : Code de l'agence bancaire ou de la succursale de rattachement du client.
*   **`age`** : Âge du client en années.
*   **`ed`** : Niveau d'études atteint par le client.
*   **`employ`** : Ancienneté professionnelle (nombre d'années passées chez l'employeur actuel).
*   **`address`** : Stabilité résidentielle (nombre d'années passées à l'adresse actuelle).
*   **`income`** : Revenu annuel du client (exprimé en milliers d'euros).
*   **`debtinc`** (*Debt-to-Income ratio*) : Taux d'endettement global du client (en %).
*   **`creddebt`** (*Credit Debt*) : Encours de la dette liée aux cartes de crédit et crédits conso (en milliers d'euros).
*   **`othdebt`** (*Other Debt*) : Encours des autres dettes bancaires ou privées (en milliers d'euros).
*   **`default`** (Target) : Statut de défaut de paiement du client (`Oui` = en défaut / `Non` = a remboursé).

In [ ]:
# Afficher le résumé statistique des variables numériques
df.describe()

In [ ]:
# Résumé des variables catégorielles
df.describe(include=["object", "category"])

In [ ]:
# Affiche les pourcentages des var catégorielles (ex: Bac+2  35.16)
logging.info(df['ed'].value_counts(normalize=True) * 100)

logging.info(df['default'].value_counts(normalize=True) * 100)

In [ ]:
df.info()

## Prétraitement des données

### Nettoyage

In [ ]:
logging.info(f"Nombre de valeurs manquantes : {df.isnull().sum().sum()}")  # Vérifie les valeurs manquantes dans le dataset

data_cleaned = df.dropna()  # Supprime les lignes avec des valeurs manquantes
logging.info(f"Dimensions du dataset après suppression des valeurs manquantes : {data_cleaned.shape}")

data_cleaned = data_cleaned.drop_duplicates()  # Supprime les doublons
logging.info(f"Dimensions du dataset après suppression des doublons : {data_cleaned.shape}")

data_cleaned = data_cleaned.drop(columns=['ncust', 'customer'])  # Supprime la colonne 'id' si elle existe
logging.info(f"Dimensions du dataset après suppression des colonnes 'ncust' et 'customer' : {data_cleaned.shape}")

In [ ]:
data_cleaned

### Structuration et Typage des Données

Afin de garantir le bon comportement de nos futurs modèles de Machine Learning (notamment la gestion des distances pour les modèles linéaires et les séparations pour les arbres), les types de données du dataset nettoyé sont standardisés comme suit :

#### Variables Numériques Entières (`int64`)
Ces variables représentent des comptes discrets (durées ou âges, toutes sont des années) :
*   **`age`** : Âge du client en années.
*   **`address`** : Nombre d'années passées à l'adresse actuelle.
*   **`employ`** : Ancienneté chez l'employeur actuel (en années).
*   **`default`** : Statut de défaut de paiement (Variable cible / *Target*) à transformer en 0 et 1 au lieu de Non et Oui

#### Variables Numériques Continues (`float64`)
Ces variables représentent des montants financiers ou des ratios :
*   **`income`** : Revenu annuel du client (en k€).
*   **`debtinc`** : Taux d'endettement global (en %).
*   **`creddebt`** : Encours de la dette liée aux cartes de crédit (en k€).
*   **`othdebt`** : Encours des autres dettes bancaires (en k€).

#### Variables Catégorielles Nominales (`category`)
*   **`branch`** : Code de l'agence bancaire de rattachement.

#### Variable Catégorielle Ordinale (Encodée en `int64`)
*   **`ed`** : Niveau d'études atteint.
    > **Stratégie de Feature Engineering :** Initialement textuelle, cette variable est convertie en valeurs numériques ordonnées ($1$ à $5$) afin de préserver la hiérarchie logique des diplômes tout en facilitant son interprétation par les algorithmes.

### Target/Label mapping en 0 et 1

In [ ]:
# Définir le mapping ('Oui' = 1 ; 'Non' = 0)
target_mapping = {'Oui': 1, 'Non': 0}

# Appliquer la transformation sur la colonne 'default' pour convertir les valeurs en 0 et 1
data_cleaned['default'] = data_cleaned['default'].map(target_mapping)

# Vérification rapide pour valider le type (int)
print(data_cleaned['default'].value_counts())

### Autres mapping et changement de type

In [ ]:
# Check visuel du jeu de données dans les logs
logging.info(
    f"Aperçu des premières lignes pour validation :\n{data_cleaned[['age', 'ed', 'income']].head()}"
)

# Définir le mapping logique pour convertir les chaînes en nombres ordonnés
ed_mapping = {
    "Niveau bac": 1,
    "Bac+2": 2,
    "Bac+3": 3,
    "Bac+4": 4,
    "Bac+5 et plus": 5,
}

# Appliquer le mapping sur la colonne 'ed'
data_cleaned["ed"] = data_cleaned["ed"].map(ed_mapping)

# On vérifie s'il y a eu des ratés (ex: une modalité mal orthographiée qui serait devenue NaN)
if data_cleaned["ed"].isnull().any():
    logging.warning(
        "Attention : Certaines valeurs de 'ed' n'ont pas pu être mappées et sont devenues NaN !"
    )

# 3. Conversion globale de tous les types (incluant 'ed' en int64 maintenant)
data_cleaned = data_cleaned.astype(
    {
        # Numériques entiers (avec 'ed' qui rejoint le groupe)
        "age": "int64",
        "address": "int64",
        "employ": "int64",
        "ed": "int64",  # <--- Devient un entier ordonné
        "default": "int64", # le label de sortie (0 ou 1)
        # Numériques continus
        "income": "float64",
        "debtinc": "float64",
        "creddebt": "float64",
        "othdebt": "float64",
        # Catégorielles nominales
        "branch": "category",
    }
)

logging.info(
    "Mapping et conversion des types effectués. 'ed' est maintenant un entier ordonné (1 à 5)."
)

# 4. Petit check visuel dans les logs pour valider le mapping
logging.info(
    f"Aperçu des premières lignes pour validation :\n{data_cleaned[['age', 'ed', 'income']].head()}"
)

In [ ]:
data_cleaned.info()

### Découpage dataset train, test, pred

In [ ]:
# Création sécurisée du dossier de sortie
output_dir = os.path.join(path_data, "output")
os.makedirs(output_dir, exist_ok=True)

# On isole d'abord un échantillon pour simuler les futurs clients (data_pred)
# Ici on prend 50 lignes au hasard
data_pred_raw = data_cleaned.sample(n=50, random_state=42)

# On isole et on garde le label (y_pred) de côté
data_pred_labels = data_pred_raw[['default']].copy()

# On crée le jeu de features (X_pred) sans la cible pour simuler le réel
data_pred = data_pred_raw.drop(columns=['default'])

# Le reste des données sert au Train et au Test
# On retire donc les lignes de data_pred_raw du dataset principal
data_modeling = data_cleaned.drop(data_pred_raw.index)

# Séparation en Train (80%) et Test (20%) avec stratification
data_train, data_test = train_test_split(
    data_modeling, 
    test_size=0.20, 
    random_state=42, 
    stratify=data_modeling['default']
)

# Logs de validation
logging.info(f"Séparation des données terminée.")
logging.info(f"Dimensions de data_train (Entraînement) : {data_train.shape}")
logging.info(f"Dimensions de data_test (Validation R&D) : {data_test.shape}")
logging.info(f"Dimensions de data_pred (Futurs clients à scorer - sans target) : {data_pred.shape}")

# 6. Sauvegarde des fichiers au format CSV
data_train.to_csv(os.path.join(output_dir, "data_train.csv"), index=False)
data_test.to_csv(os.path.join(output_dir, "data_test.csv"), index=False)
data_pred.to_csv(os.path.join(output_dir, "data_pred.csv"), index=False)
data_pred_labels.to_csv(os.path.join(output_dir, "data_pred_labels.csv"), index=False)

logging.info(f"Les 3 datasets ont été exportés proprement dans : {output_dir}/")

In [ ]:
# --- VÉRIFICATION DES PROPORTIONS DE LA TARGET (STRATIFICATION) ---
# Calcul direct du pourcentage de défauts
pct_train = (data_train["default"] == 1).mean() * 100
pct_test = (data_test["default"] == 1).mean() * 100
pct_pred = (data_pred_labels["default"] == 1).mean() * 100

# Envoi des métriques dans le logger
logging.info(
    "Vérification de la distribution de la variable cible (default = 1) :"
)
logging.info(f" -> Proportion dans le jeu d'entraînement (Train) : {pct_train:.2f}%")
logging.info(f" -> Proportion dans le jeu de validation (Test)    : {pct_test:.2f}%")
logging.info(f" -> Proportion dans le jeu simulé de prod (Pred)   : {pct_pred:.2f}%")

### Visualisation des données préparées

In [ ]:
# Distribution de chaque variable, valeurs manquantes, taux de doublons
# et matrices de corrélation entre tes features et ta target.
profile = ProfileReport(data_train, title="Rapport d'Exploration Données")
profile.to_notebook_iframe()

In [ ]:
# Comparer les deux datasets train et test pour vérifier la stratification et les distributions
report = sv.compare([data_train, "Train"], [data_test, "Test"], target_feat='default')
report.show_notebook()

### Entrainement

#### Méthodologie d'Entraînement et de Validation
La stratégie de modélisation et de sélection du modèle champion suit une approche rigoureuse en trois phases :

1. Phase de R&D (Entraînement et Optimisation)
*   **Multi-modèles & Grid Search :** Chargement de plusieurs architectures de classification (ex: Régression Logistique, Random Forest, XGBoost etc...) avec leurs grilles de reconfigurations (hyperparamètres) respectives.
*   **Validation Croisée (Cross-Validation) :** Chaque configuration est entraînée en validation croisée afin de garantir la robustesse des résultats et d'éviter le surapprentissage (*overfitting*).
*   **Métrique de sélection initiale :** Le meilleur candidat de chaque famille de classifieurs est sélectionné sur la base de l'**AUC-PR** (Précision-Rappel), particulièrement adaptée aux jeux de données déséquilibrés. 
*   **Suivi Exhaustif :** L'ensemble des métriques métier et techniques (Précision, Rappel, F2-Score, AUC-ROC) définies dans le cahier des charges (`README.md`) sont calculées et historisées pour analyse.


2. Sélection du Modèle Champion
*   **Arbitrage Métier :** Le modèle vainqueur final (le "Champion") est choisi parmi les meilleurs candidats sur le critère du **F2-Score** afin de maximiser le **Rappel (Recall)** tout en conservant un arbitrage minimal sur la Précision.
    > 🎯 **Justification Métier :** En octroi de crédit, le coût financier d'un faux négatif (valider un client qui va faire défaut) est largement supérieur au coût d'un faux positif (refuser un client qui aurait remboursé). Maximiser le Rappel via le F2-Score permet de capturer un maximum de profils à risque sans pour autant dégrader aveuglément la sélectivité globale du modèle.

3. Phase de Déploiement (Simulation de Production / PRED)
Le modèle champion est appliqué sur le jeu de données indépendant `data_pred` selon les règles suivantes :
*   **XAI (Explainable AI) :** Calcul et analyse des **SHAP values** sur les prévisions pour garantir l'explicabilité locale des scores de crédit attribués.
*   **Évaluation de la performance (Ground Truth) :** Profitant de la disponibilité des labels réels isolés dans `data_pred_labels`, les performances du modèle en conditions réelles peuvent être mesurées.
*   **Flexibilité du Pipeline :** Intégration d'un paramètre d'exécution conditionnel (booléen) permettant de basculer entre deux modes :
    1.  *Mode Standard :* Génération et exposition des prévisions seules (scénario de production pure).
    2.  *Mode Évaluation :* Calcul et logging des métriques de performance globales (scénario d'audit ou de monitoring de dérive).